In [ ]:
import os
import time
import csv
import hashlib
import base58
from Crypto.Hash import RIPEMD160
from google.colab import files
import random
import logging
from tqdm import tqdm

# Install minimal dependencies
!pip install pycryptodome base58 tqdm --force-reinstall --no-cache-dir --quiet

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# Setup checkpoint directory
CHECKPOINT_DIR = '/content/ripemphantom_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Upload addresses.txt
logger.info("Upload addresses.txt with 1MVLP2kRPNqz8VJUy83LstUoMQzUjgq4Zg")
files.upload()
ADDRESS_FILE = 'addresses.txt'

# Constants
TARGET_ADDRESS = '1MVLP2kRPNqz8VJUy83LstUoMQzUjgq4Zg'
SEED_KEY = 0x4197113
HAMMING_THRESHOLD = 45
BEST_HAMMING = 39
INITIAL_RANGE = 10000000
MAX_KEY = 2**61
CHECKPOINT_INTERVAL = 10
MIN_HAMMING_FOR_MUTATION = 35
STAGNATION_THRESHOLD = 10
BATCH_SIZE = 10000

# Files
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'checkpoint.csv')
OUTPUT_CSV = os.path.join(CHECKPOINT_DIR, 'new_matches.csv')

# Helper functions
def privkey_to_pubkey(privkey):
    privkey_bytes = privkey.to_bytes(32, 'big')
    return hashlib.sha256(privkey_bytes).digest()

def pubkey_to_address(pubkey):
    sha256_hash = hashlib.sha256(pubkey).digest()
    ripemd160 = RIPEMD160.new()
    ripemd160.update(sha256_hash)
    hash160 = ripemd160.digest()
    versioned = b'\x00' + hash160
    checksum = hashlib.sha256(hashlib.sha256(versioned).digest()).digest()[:4]
    return base58.b58encode(versioned + checksum).decode()

def hamming_distance(hash1, hash2):
    return bin(int.from_bytes(hash1, 'big') ^ int.from_bytes(hash2, 'big')).count('1')

def hash160_from_address(address):
    try:
        decoded = base58.b58decode(address)
        return decoded[1:-4]
    except Exception as e:
        logger.error(f"Invalid address {address}: {e}")
        raise

# Hardcoded CSV
def read_csv(file_path):
    results = []
    if not os.path.exists(file_path):
        return results
    with open(file_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                results.append({
                    'Address': row['Address'],
                    'Private Key': row['Private Key'],
                    'Hamming Distance': float(row['Hamming Distance']),
                    'Similarity %': float(row['Similarity %']),
                    'Entropy Level': float(row['Entropy Level']),
                    'Exact Match': row['Exact Match'].lower() == 'true'
                })
            except:
                pass
    return results

def write_csv(file_path, results):
    with open(file_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['Address', 'Private Key', 'Hamming Distance', 'Similarity %', 'Entropy Level', 'Exact Match'])
        writer.writeheader()
        for result in results:
            writer.writerow(result)

# Bit flip heuristic
def get_flip_positions(key, target_hash160, current_hash160):
    xor = int.from_bytes(current_hash160, 'big') ^ int.from_bytes(target_hash160, 'big')
    bits = [int(b) for b in bin(xor)[2:].zfill(160)]
    return [i for i, b in enumerate(bits) if b == 1][:10]

# Process batch
def process_batch(batch, target_hash160, min_hamming):
    results = []
    mutation_mode = min_hamming < MIN_HAMMING_FOR_MUTATION
    for key in batch:
        pubkey = privkey_to_pubkey(key)
        ripemd160 = RIPEMD160.new()
        ripemd160.update(hashlib.sha256(pubkey).digest())
        candidate_hash160 = ripemd160.digest()
        hamming = hamming_distance(candidate_hash160, target_hash160)
        if hamming <= HAMMING_THRESHOLD:
            address = pubkey_to_address(pubkey)
            similarity = (160 - hamming) / 160 * 100
            result = {
                'Address': address,
                'Private Key': hex(key),
                'Hamming Distance': hamming,
                'Similarity %': similarity,
                'Entropy Level': 0.0,
                'Exact Match': hamming == 0
            }
            results.append(result)
            if hamming < BEST_HAMMING:
                print(f"NEW BEST: Key {hex(key)}, Hamming {hamming}")
                logger.info(f"New best: Key {hex(key)}, Hamming {hamming}")
        if mutation_mode or hamming < BEST_HAMMING:
            flip_positions = get_flip_positions(key, target_hash160, candidate_hash160)
            for pos in flip_positions:
                flipped_key = key ^ (1 << pos)
                if 1 <= flipped_key < MAX_KEY:
                    flipped_pubkey = privkey_to_pubkey(flipped_key)
                    ripemd160 = RIPEMD160.new()
                    ripemd160.update(hashlib.sha256(flipped_pubkey).digest())
                    flipped_hash160 = ripemd160.digest()
                    flipped_hamming = hamming_distance(flipped_hash160, target_hash160)
                    if flipped_hamming <= HAMMING_THRESHOLD:
                        flipped_address = pubkey_to_address(flipped_pubkey)
                        flipped_similarity = (160 - flipped_hamming) / 160 * 100
                        result = {
                            'Address': flipped_address,
                            'Private Key': hex(flipped_key),
                            'Hamming Distance': flipped_hamming,
                            'Similarity %': flipped_similarity,
                            'Entropy Level': 0.0,
                            'Exact Match': flipped_hamming == 0
                        }
                        results.append(result)
                        if flipped_hamming < BEST_HAMMING:
                            print(f"NEW BEST (BIT FLIP): Key {hex(flipped_key)}, Hamming {flipped_hamming}")
                            logger.info(f"Bit flip success: Key {hex(flipped_key)}, Hamming {flipped_hamming}")
    return results

# Load target address
try:
    with open(ADDRESS_FILE, 'r') as f:
        addresses = [line.strip() for line in f if line.strip()]
    if len(addresses) != 999 or TARGET_ADDRESS not in addresses:
        logger.error(f"Expected 999 addresses including {TARGET_ADDRESS}")
        raise ValueError("Invalid addresses.txt")
    target_hash160 = hash160_from_address(TARGET_ADDRESS)
except FileNotFoundError:
    logger.error(f"{ADDRESS_FILE} not found")
    raise
except Exception as e:
    logger.error(f"Error loading addresses: {e}")
    raise

# Initialize results
results = read_csv(CHECKPOINT_FILE)
if results:
    logger.info(f"Loaded {len(results)} results from checkpoint")

# Search loop
current_key = SEED_KEY
search_range = INITIAL_RANGE
iteration = 0
start_time = time.time()
last_checkpoint = start_time
min_hamming = BEST_HAMMING
stagnation_count = 0

while current_key < MAX_KEY:
    try:
        if stagnation_count >= STAGNATION_THRESHOLD:
            current_key = min(max(1, SEED_KEY + random.randint(-10**9, 10**9)), MAX_KEY - 1)
            search_range = INITIAL_RANGE
            stagnation_count = 0
            logger.info(f"Stagnation detected, jumping to key {hex(current_key)}")
        keys = list(range(max(1, current_key - search_range), current_key + search_range + 1))
        keys = [k for k in keys if 1 <= k < MAX_KEY]
        batches = [keys[i:i + BATCH_SIZE] for i in range(0, len(keys), BATCH_SIZE)]
        for batch in tqdm(batches, desc=f"Iteration {iteration}, Keys {current_key-search_range} to {current_key+search_range}"):
            batch_result = process_batch(batch, target_hash160, min_hamming)
            results.extend(batch_result)
            new_min_hamming = min([r['Hamming Distance'] for r in batch_result] + [min_hamming])
            if new_min_hamming < min_hamming:
                min_hamming = new_min_hamming
                stagnation_count = 0
                logger.info(f"Updated min Hamming: {min_hamming}")
            else:
                stagnation_count += 1
        current_time = time.time()
        if results and (current_time - last_checkpoint >= CHECKPOINT_INTERVAL):
            write_csv(OUTPUT_CSV, results)
            write_csv(CHECKPOINT_FILE, results)
            logger.info(f"Saved {len(results)} results to {OUTPUT_CSV}")
            last_checkpoint = current_time
        if min_hamming < BEST_HAMMING:
            search_range = int(search_range * 0.7)
            logger.info(f"Improved Hamming ({min_hamming}), shrinking range to ±{search_range}")
        else:
            search_range = int(search_range * 1.5)
            logger.info(f"No improvement, growing range to ±{search_range}")
        current_key += search_range
        iteration += 1
        elapsed = current_time - start_time
        logger.info(f"Iteration {iteration}: Tested {current_key}, Range ±{search_range}, Min Hamming {min_hamming}, Elapsed {elapsed:.2f}s")
        if any(r['Exact Match'] for r in results):
            print("EXACT MATCH FOUND!")
            logger.info("Exact match found! Stopping search.")
            write_csv(OUTPUT_CSV, results)
            write_csv(CHECKPOINT_FILE, results)
            break
    except Exception as e:
        logger.error(f"Error in iteration {iteration}: {e}")
        time.sleep(5)

logger.info("Search done. Check /content/ripemphantom_checkpoints")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 303.7 MB/s eta 0:00:00


Saving addresses.txt to addresses.txt


Iteration 708, Keys 605336072 to 625336072:  44%|████▍     | 884/2001 [01:06<01:24, 13.28it/s]

In [ ]:
# Install required packages:
# pip install pycryptodome base58 ecdsa numpy

import hashlib
from Crypto.Hash import RIPEMD160
import base58
import numpy as np
import csv
from ecdsa import SigningKey, SECP256k1

# Maximum SHA-256 passes to test
MAX_PASSES = 3
# Modulo levels to test
MODULO_LEVELS = [256, 512, 1024, 2048, 4096]
# Key range (100,000 keys, around 0x006d816 = 447510)
START_KEY = 0x0060000  # 393,216
END_KEY = 0x00786a0   # 493,216

# Read target addresses from addresses.txt
def load_target_addresses(file_path="addresses.txt"):
    try:
        with open(file_path, 'r') as f:
            addresses = [line.strip() for line in f if line.strip()]
        return addresses
    except FileNotFoundError:
        raise FileNotFoundError("addresses.txt not found. Please provide the file with 999 target addresses.")

# Decode address to get RIPEMD-160 hash
def get_ripemd160_from_address(address):
    try:
        decoded = base58.b58decode_check(address)
        return decoded[1:]  # Strip version byte
    except ValueError:
        return None

# Compute Hamming distance between two byte arrays
def hamming_distance(bytes1, bytes2):
    return np.sum([bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2)])

# Compute chi-squared statistic for RIPEMD-160 hash
def chi_squared(hash_bytes):
    observed = np.array([b for b in hash_bytes])
    expected = 128  # Uniform expected value (256/2)
    chi2 = np.sum((observed - expected) ** 2 / expected)
    return chi2

# Generate Bitcoin address from private key
def private_key_to_address(priv_key_int, sha256_passes=1, compressed=True):
    priv_key_bytes = priv_key_int.to_bytes(32, 'big')
    sk = SigningKey.from_string(priv_key_bytes, curve=SECP256k1)
    pub_key = sk.get_verifying_key().to_string("compressed" if compressed else "uncompressed")
    h = pub_key
    for i in range(sha256_passes):
        h = hashlib.sha256(h).digest()
        print(f"Debug: Key {hex(priv_key_int)[2:].zfill(64)}, SHA-256 pass {i+1}/{sha256_passes}, Format {'compressed' if compressed else 'uncompressed'}, Hash: {h.hex()}, Length: {len(h)}")
    ripemd160 = RIPEMD160.new()
    ripemd160.update(h)
    hashed = ripemd160.digest()
    versioned = b'\x00' + hashed
    checksum = hashlib.sha256(hashlib.sha256(versioned).digest()).digest()[:4]
    address = base58.b58encode(versioned + checksum)
    return address.decode(), hashed

# Main function to test keys against multiple addresses
def main():
    output_file = "key_recovery_multi_sha_results.csv"
    with open(output_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Private_Key_Hex", "Target_Address", "SHA256_Passes", "Key_Format", "Hamming_Distance", "Chi_Squared", "Modulo_Level", "Modulo_Value"])

    # Load target addresses
    target_addresses = load_target_addresses()
    target_ripemd160s = {addr: get_ripemd160_from_address(addr) for addr in target_addresses}
    target_ripemd160s = {addr: ripemd for addr, ripemd in target_ripemd160s.items() if ripemd is not None}

    # Track global best chi-squared per address, pass count, key format, and modulo level
    global_best = {
        addr: {
            p: {
                fmt: {
                    m: {"distance": float('inf'), "chi2": float('inf'), "key": None, "address": None, "modulo": None}
                    for m in MODULO_LEVELS
                }
                for fmt in ["compressed", "uncompressed"]
            }
            for p in range(1, MAX_PASSES + 1)
        }
        for addr in target_ripemd160s
    }

    # Test keys in range
    for key_int in range(START_KEY, END_KEY):
        key_hex = hex(key_int)[2:].zfill(64)
        for passes in range(1, MAX_PASSES + 1):
            for compressed in [True, False]:
                key_format = "compressed" if compressed else "uncompressed"
                address, ripemd160 = private_key_to_address(key_int, passes, compressed)
                for target_addr, target_ripemd in target_ripemd160s.items():
                    distance = hamming_distance(ripemd160, target_ripemd)
                    chi2 = chi_squared(ripemd160)
                    # Update global best for each modulo level
                    for modulo_level in MODULO_LEVELS:
                        modulo_value = key_int % modulo_level
                        if chi2 < global_best[target_addr][passes][key_format][modulo_level]["chi2"]:
                            global_best[target_addr][passes][key_format][modulo_level] = {
                                "distance": distance, "chi2": chi2, "key": key_hex, "address": address, "modulo": modulo_value
                            }
                            print(f"New best for {target_addr}, {passes} SHA-256 passes, {key_format}, Modulo {modulo_level}={modulo_value}: Key {key_hex}, Address {address}, Hamming {distance}, Chi2 {chi2:.2f}")
                        # Stop if Hamming distance is 0
                        if distance == 0:
                            print(f"Exact match found! Key {key_hex}, Target {target_addr}, SHA-256 passes {passes}, Format {key_format}, Chi2 {chi2:.2f}, Modulo {modulo_level}={modulo_value}")
                            return
                        # Save all results to CSV
                        with open(output_file, 'a', newline='') as f:
                            writer = csv.writer(f)
                            writer.writerow([key_hex, target_addr, passes, key_format, distance, chi2, modulo_level, modulo_value])

    # Print final best results
    print("\nFinal Best Chi-Squared Values:")
    for target_addr in target_ripemd160s:
        for passes in range(1, MAX_PASSES + 1):
            for key_format in ["compressed", "uncompressed"]:
                for modulo_level in MODULO_LEVELS:
                    data = global_best[target_addr][passes][key_format][modulo_level]
                    if data["key"] is None:
                        print(f"{target_addr}, {passes} SHA-256 passes, {key_format}, Modulo {modulo_level}: No keys tested")
                    else:
                        print(f"{target_addr}, {passes} SHA-256 passes, {key_format}: Best key is {data['key']} at {data['distance']} Hamming, Chi2 {data['chi2']:.2f}, Modulo_Level {modulo_level}, Modulo_Value {data['modulo']}, Address {data['address']}")

if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.
New best for 1PyfQYtsPnGd6ohu1HVHnT2HigCicF7Acp, 1 SHA-256 passes, uncompressed, Modulo 256=112: Key 0000000000000000000000000000000000000000000000000000000000060f70, Address 1EXkYr13PuELRKZfp5GHmq2mPuJ4eerBgy, Hamming 73, Chi2 138.44
New best for 1PyfQYtsPnGd6ohu1HVHnT2HigCicF7Acp, 1 SHA-256 passes, uncompressed, Modulo 512=368: Key 0000000000000000000000000000000000000000000000000000000000060f70, Address 1EXkYr13PuELRKZfp5GHmq2mPuJ4eerBgy, Hamming 73, Chi2 138.44
New best for 1PyfQYtsPnGd6ohu1HVHnT2HigCicF7Acp, 1 SHA-256 passes, uncompressed, Modulo 1024=880: Key 0000000000000000000000000000000000000000000000000000000000060f70, Address 1EXkYr13PuELRKZfp5GHmq2mPuJ4eerBgy, Hamming 73, Chi2 138.44
New best for 1PyfQYtsPnGd6ohu1HVHnT2HigCicF7Acp, 1 SHA-256 passes, uncompressed, Modulo 2048=1904: Key 0000000000000000000000000000000000000000000000000000000000060f70, Address 1EXkYr13PuELRKZfp5GHmq2mPuJ4eerBgy, Hamming 73, Chi2 138.44
New be

In [ ]:
# Install required packages:
# pip install pycryptodome base58 ecdsa numpy

import hashlib
from Crypto.Hash import RIPEMD160
import base58
import numpy as np
import csv
from ecdsa import SigningKey, SECP256k1

# Maximum SHA-256 passes to test
MAX_PASSES = 3
# Modulo levels to test
MODULO_LEVELS = [256, 512, 1024, 2048, 4096]
# Priority modulo values for console logging (from 0x006d816=22, 0x059ee05f=95, plus 0, 128, 160)
MODULO_PRIORITY = [22, 95, 0, 128, 160]
# Key range (100,000 keys, around 0x006d816 = 447510)
START_KEY = 0x0060000  # 393,216
END_KEY = 0x00786a0   # 493,216

# Read target addresses from addresses.txt
def load_target_addresses(file_path="addresses.txt"):
    try:
        with open(file_path, 'r') as f:
            addresses = [line.strip() for line in f if line.strip()]
        return addresses
    except FileNotFoundError:
        raise FileNotFoundError("addresses.txt not found. Please provide the file with 999 target addresses.")

# Decode address to get RIPEMD-160 hash
def get_ripemd160_from_address(address):
    try:
        decoded = base58.b58decode_check(address)
        return decoded[1:]  # Strip version byte
    except ValueError:
        return None

# Compute Hamming distance between two byte arrays
def hamming_distance(bytes1, bytes2):
    return np.sum([bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2)])

# Compute chi-squared statistic for RIPEMD-160 hash
def chi_squared(hash_bytes):
    observed = np.array([b for b in hash_bytes])
    expected = 128  # Uniform expected value (256/2)
    chi2 = np.sum((observed - expected) ** 2 / expected)
    return chi2

# Generate Bitcoin address from private key
def private_key_to_address(priv_key_int, sha256_passes=1, compressed=True):
    priv_key_bytes = priv_key_int.to_bytes(32, 'big')
    sk = SigningKey.from_string(priv_key_bytes, curve=SECP256k1)
    pub_key = sk.get_verifying_key().to_string("compressed" if compressed else "uncompressed")
    h = pub_key
    for i in range(sha256_passes):
        h = hashlib.sha256(h).digest()
        print(f"Debug: Key {hex(priv_key_int)[2:].zfill(64)}, SHA-256 pass {i+1}/{sha256_passes}, Format {'compressed' if compressed else 'uncompressed'}, Hash: {h.hex()}, Length: {len(h)}")
    ripemd160 = RIPEMD160.new()
    ripemd160.update(h)
    hashed = ripemd160.digest()
    versioned = b'\x00' + hashed
    checksum = hashlib.sha256(hashlib.sha256(versioned).digest()).digest()[:4]
    address = base58.b58encode(versioned + checksum)
    return address.decode(), hashed

# Main function to test keys against multiple addresses
def main():
    output_file = "key_recovery_multi_sha_results.csv"
    with open(output_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Private_Key_Hex", "Target_Address", "SHA256_Passes", "Key_Format", "Hamming_Distance", "Chi_Squared", "Modulo_Level", "Modulo_Value"])

    # Load target addresses
    target_addresses = load_target_addresses()
    target_ripemd160s = {addr: get_ripemd160_from_address(addr) for addr in target_addresses}
    target_ripemd160s = {addr: ripemd for addr, ripemd in target_ripemd160s.items() if ripemd is not None}

    # Track global best chi-squared per address, pass count, key format, and modulo level
    global_best = {
        addr: {
            p: {
                fmt: {
                    m: {"distance": float('inf'), "chi2": float('inf'), "key": None, "address": None, "modulo": None}
                    for m in MODULO_LEVELS
                }
                for fmt in ["compressed", "uncompressed"]
            }
            for p in range(1, MAX_PASSES + 1)
        }
        for addr in target_ripemd160s
    }

    # Test keys in range
    for key_int in range(START_KEY, END_KEY):
        key_hex = hex(key_int)[2:].zfill(64)
        modulo_values = {m: key_int % m for m in MODULO_LEVELS}
        log_priority = modulo_values[256] in MODULO_PRIORITY
        for passes in range(1, MAX_PASSES + 1):
            for compressed in [True, False]:
                key_format = "compressed" if compressed else "uncompressed"
                address, ripemd160 = private_key_to_address(key_int, passes, compressed)
                for target_addr, target_ripemd in target_ripemd160s.items():
                    distance = hamming_distance(ripemd160, target_ripemd)
                    chi2 = chi_squared(ripemd160)
                    # Update global best for each modulo level
                    for modulo_level in MODULO_LEVELS:
                        modulo_value = modulo_values[modulo_level]
                        if chi2 < global_best[target_addr][passes][key_format][modulo_level]["chi2"]:
                            global_best[target_addr][passes][key_format][modulo_level] = {
                                "distance": distance, "chi2": chi2, "key": key_hex, "address": address, "modulo": modulo_value
                            }
                            if log_priority or chi2 < 100:
                                print(f"New best for {target_addr}, {passes} SHA-256 passes, {key_format}, Modulo {modulo_level}={modulo_value}: Key {key_hex}, Address {address}, Hamming {distance}, Chi2 {chi2:.2f}")
                        # Stop if Hamming distance is 0
                        if distance == 0:
                            print(f"Exact match found! Key {key_hex}, Target {target_addr}, SHA-256 Passes {passes}, Format {key_format}, Chi2 {chi2:.2f}, Modulo {modulo_level}={modulo_value}")
                            return
                        # Save all results to CSV
                        with open(output_file, 'a', newline='') as f:
                            writer = csv.writer(f)
                            writer.writerow([key_hex, target_addr, passes, key_format, distance, chi2, modulo_level, modulo_value])

    # Print final best results
    print("\nFinal Best Chi-Squared Values:")
    for target_addr in target_ripemd160s:
        for passes in range(1, MAX_PASSES + 1):
            for key_format in ["compressed", "uncompressed"]:
                for modulo_level in MODULO_LEVELS:
                    data = global_best[target_addr][passes][key_format][modulo_level]
                    if data["key"] is None:
                        print(f"{target_addr}, {passes} SHA-256 passes, {key_format}, Modulo {modulo_level}: No keys tested")
                    else:
                        print(f"{target_addr}, {passes} SHA-256 passes, {key_format}: Best key is {data['key']} at {data['distance']} Hamming, Chi2 {data['chi2']:.2f}, Modulo_Level {modulo_level}, Modulo_Value {data['modulo']}, Address {data['address']}")

if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.
Debug: Key 000000000000000000000000000000000000000000000000000000000006055d, SHA-256 pass 3/3, Format compressed, Hash: 23ffba532dac59bd3cb5d689565870d8bcfc60a41333ef5a31808d556abc2c3a, Length: 32
Debug: Key 000000000000000000000000000000000000000000000000000000000006055d, SHA-256 pass 1/3, Format uncompressed, Hash: d31b504d5cbd67046058feb1aeded017b9e24bec10eb49c7f11dac5dab3f67af, Length: 32
Debug: Key 000000000000000000000000000000000000000000000000000000000006055d, SHA-256 pass 2/3, Format uncompressed, Hash: e42fa5272baf399f9324c62f44100686f6724e384a94a9c146b8c3dc3a2bb719, Length: 32
Debug: Key 000000000000000000000000000000000000000000000000000000000006055d, SHA-256 pass 3/3, Format uncompressed, Hash: 2e8715f64f495743aa43699acf1417364083cf1052c1161ae31c7a2911e57aec, Length: 32
Debug: Key 000000000000000000000000000000000000000000000000000000000006055e, SHA-256 pass 1/1, Format compressed, Hash: 64e4a1f53e73f7f8937a8e0d1078d42f4b1

KeyboardInterrupt: 

In [ ]:
import os
import time
import random
import hashlib
import base58
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
from collections import defaultdict, deque
from multiprocessing import Pool, cpu_count, Manager
from tqdm import tqdm
import numpy as np
from scipy.stats import chi2_contingency
import pickle

# Constants
MAX_K = 2**256
TARGETS = []
CHECKPOINT_FILE = "recovery_checkpoint.pkl"
STATE_FILE = "recovery_state.pkl"
STAGNATION_THRESHOLD = 50000
TEMPERATURE = 1000.0
COOLING_RATE = 0.999
CANDIDATE_LIMIT = 10

# Key-to-Hash160
def privkey_to_hash160(k: int, compressed: bool = True) -> bytes:
    try:
        if not isinstance(k, int) or not (1 <= k < SECP256k1.order):
            return None
        sk = SigningKey.from_secret_exponent(k, curve=SECP256k1)
        vk = sk.get_verifying_key()
        if compressed:
            prefix = b'\x02' if vk.to_string()[32] % 2 == 0 else b'\x03'
            pubkey = prefix + vk.to_string()[:32]
        else:
            pubkey = b'\x04' + vk.to_string()
        sha = hashlib.sha256(pubkey).digest()
        ripemd = RIPEMD160.new(); ripemd.update(sha)
        return ripemd.digest()
    except Exception:
        return None

# Bit Hamming Distance
def hamming_distance_bits(hash1: bytes, hash2: bytes) -> tuple[int, list[int]]:
    distance = 0
    flips = []
    for i in range(len(hash1)):
        xor = hash1[i] ^ hash2[i]
        for j in range(8):
            if xor & (1 << j):
                bit_index = (i * 8) + (7 - j)
                flips.append(bit_index)
                distance += 1
    return distance, flips

# Address Generation
def hash160_to_address(h160: bytes) -> str:
    version = b'\x00'
    payload = version + h160
    checksum = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
    return base58.b58encode(payload + checksum).decode()

# Load Targets
def load_targets():
    try:
        with open('addresses.txt', 'r') as f:
            for line in f:
                addr = line.strip()
                if not addr:
                    continue
                try:
                    decoded = base58.b58decode(addr)
                    h160 = decoded[1:-4]
                    if len(h160) == 20:
                        TARGETS.append((addr, h160))
                except:
                    continue
        print(f"Loaded {len(TARGETS)} addresses from addresses.txt.")
        if not TARGETS:
            print("Error: No valid addresses loaded. Exiting.")
            exit(1)
    except FileNotFoundError:
        print("Error: addresses.txt not found. Please upload the file. Exiting.")
        exit(1)

# Chi-Squared Anomaly Detection
def chi_squared_anomaly(hamming_history: list) -> float:
    if len(hamming_history) < 100:
        return 1.0
    observed = np.histogram(hamming_history, bins=20, range=(0, 160))[0]
    expected = np.ones(20) * len(hamming_history) / 20
    try:
        chi2, p = chi2_contingency([observed, expected])[:2]
        return p
    except:
        return 1.0

# Update Strategy Weights
def update_strategy_weights(strategy: str, success: float, shared_state: dict, lock):
    with lock:
        if success > 0:
            shared_state['STRATEGY_WEIGHTS'][strategy] = min(shared_state['STRATEGY_WEIGHTS'][strategy] * 1.3, 0.6)
        else:
            shared_state['STRATEGY_WEIGHTS'][strategy] = max(shared_state['STRATEGY_WEIGHTS'][strategy] * 0.7, 0.05)
        total = sum(shared_state['STRATEGY_WEIGHTS'].values())
        for key in shared_state['STRATEGY_WEIGHTS']:
            shared_state['STRATEGY_WEIGHTS'][key] /= total

# Simulated Annealing Acceptance
def accept_mutation(delta_hamming: float, temperature: float) -> bool:
    if delta_hamming > 0:
        return True
    return random.random() < np.exp(delta_hamming / max(temperature, 1e-6))

# Genetic Crossover
def crossover_keys(k1: int, k2: int) -> int:
    mask = random.getrandbits(256)
    return (k1 & mask) | (k2 & ~mask)

# Mutation Engine
def mutate_key(k: int, prev_hamming: int, target_h160: bytes, target_addr: str, temperature: float, shared_state: dict, lock) -> tuple[int, int, list[int], str]:
    new_k = k
    with lock:
        strategy = random.choices(
            list(shared_state['STRATEGY_WEIGHTS'].keys()),
            list(shared_state['STRATEGY_WEIGHTS'].values())
        )[0]

    if strategy == 'bit_flip' and shared_state['BIT_BIAS']:
        with lock:
            top_bits = sorted(shared_state['BIT_BIAS'].items(), key=lambda x: -x[1])[:20]
        if top_bits:
            bit = random.choices([b[0] for b in top_bits], [b[1] for b in top_bits])[0]
            new_k ^= (1 << bit)
    elif strategy == 'xor_delta' and shared_state['MUTATION_BANK']:
        with lock:
            delta = random.choice(shared_state['MUTATION_BANK'])['delta_k']
        new_k ^= delta
    elif strategy == 'crossover' and shared_state['TOP_CANDIDATES'][target_addr]:
        with lock:
            other_k = random.choice(shared_state['TOP_CANDIDATES'][target_addr])[0]
        new_k = crossover_keys(k, other_k)
    elif strategy == 'range_shift':
        new_k = random.randint(1, MAX_K - 1)
    else:
        shift = random.randint(1, 256)
        new_k ^= random.getrandbits(shift)

    if new_k < 1 or new_k >= MAX_K:
        new_k = random.randint(1, MAX_K - 1)

    h160 = privkey_to_hash160(new_k, compressed=True) or privkey_to_hash160(new_k, compressed=False)
    if not h160:
        return k, prev_hamming, [], strategy

    ham, flips = hamming_distance_bits(h160, target_h160)
    with lock:
        shared_state['HAMMING_HISTORY'][target_addr].append(ham)
        delta_hamming = prev_hamming - ham

        if accept_mutation(delta_hamming, temperature):
            if delta_hamming > 0:
                for bit in flips:
                    shared_state['BIT_BIAS'].setdefault(bit, 0.0)
                    shared_state['BIT_BIAS'][bit] += delta_hamming * 0.2
                shared_state['MUTATION_BANK'].append({
                    'delta_hamming': delta_hamming,
                    'flips': flips,
                    'delta_k': k ^ new_k,
                    'k_old': k,
                    'k_new': new_k,
                    'addr': target_addr
                })
                shared_state['TOP_CANDIDATES'][target_addr].append((new_k, ham))
                shared_state['TOP_CANDIDATES'][target_addr] = sorted(
                    shared_state['TOP_CANDIDATES'][target_addr], key=lambda x: x[1]
                )[:CANDIDATE_LIMIT]
                update_strategy_weights(strategy, delta_hamming, shared_state, lock)
                if ham < shared_state['MIN_HAMMING'][target_addr]:
                    shared_state['MIN_HAMMING'][target_addr] = ham
                    shared_state['BEST_CANDIDATES'][target_addr] = (new_k, ham, True)
            return new_k, ham, flips, strategy
    return k, prev_hamming, [], strategy

# Checkpointing
def save_checkpoint(found_addresses: dict):
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(found_addresses, f)

def load_checkpoint():
    found = {}
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'rb') as f:
            found = pickle.load(f)
        print(f"Loaded {len(found)} previously found addresses from checkpoint.")
    return found

# Save State
def save_state(shared_state: dict, lock):
    with lock:
        state = {
            'BIT_BIAS': dict(shared_state['BIT_BIAS']),
            'MUTATION_BANK': list(shared_state['MUTATION_BANK']),
            'TOP_CANDIDATES': {k: list(v) for k, v in shared_state['TOP_CANDIDATES'].items()},
            'STRATEGY_WEIGHTS': dict(shared_state['STRATEGY_WEIGHTS']),
            'MIN_HAMMING': dict(shared_state['MIN_HAMMING']),
            'HAMMING_HISTORY': {k: list(v) for k, v in shared_state['HAMMING_HISTORY'].items()}
        }
        with open(STATE_FILE, 'wb') as f:
            pickle.dump(state, f)

def load_state(shared_state: dict, lock):
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, 'rb') as f:
            state = pickle.load(f)
        with lock:
            shared_state['BIT_BIAS'].update(state['BIT_BIAS'])
            shared_state['MUTATION_BANK'].extend(state['MUTATION_BANK'])
            for k, v in state['TOP_CANDIDATES'].items():
                shared_state['TOP_CANDIDATES'][k] = v
            shared_state['STRATEGY_WEIGHTS'].update(state['STRATEGY_WEIGHTS'])
            shared_state['MIN_HAMMING'].update(state['MIN_HAMMING'])
            for k, v in state['HAMMING_HISTORY'].items():
                shared_state['HAMMING_HISTORY'][k] = v
        print("Loaded previous state.")

# Worker Function
def worker_task(args):
    seed_k, target_addr, target_h160, shared_state, lock = args
    k = seed_k
    best_k = k
    best_ham = 160
    best_compressed = True
    temperature = TEMPERATURE
    iterations = 0

    while True:
        iterations += 1
        for compressed in [True, False]:
            h160 = privkey_to_hash160(k, compressed=compressed)
            if not h160:
                continue

            ham, flips = hamming_distance_bits(h160, target_h160)
            if ham == 0:
                return (target_addr, k, compressed, True, ham)

            if ham < best_ham:
                best_k = k
                best_ham = ham
                best_compressed = compressed
                with lock:
                    if ham < shared_state['MIN_HAMMING'][target_addr]:
                        shared_state['MIN_HAMMING'][target_addr] = ham
                        shared_state['BEST_CANDIDATES'][target_addr] = (k, ham, compressed)

        k, best_ham, flips, strategy = mutate_key(k, best_ham, target_h160, target_addr, temperature, shared_state, lock)
        temperature *= COOLING_RATE

        if iterations % 10000 == 0:
            with lock:
                p_value = chi_squared_anomaly(shared_state['HAMMING_HISTORY'][target_addr])
                if p_value < 0.05 or iterations > STAGNATION_THRESHOLD:
                    k = random.randint(1, MAX_K - 1)
                    temperature = TEMPERATURE
                    shared_state['HAMMING_HISTORY'][target_addr].clear()
                    update_strategy_weights(strategy, -1.0, shared_state, lock)

# Main Function
def main():
    load_targets()
    manager = Manager()
    lock = manager.Lock()
    shared_state = manager.dict({
        'BIT_BIAS': manager.dict(),
        'MUTATION_BANK': manager.list(),
        'TOP_CANDIDATES': manager.dict({addr: manager.list() for addr, _ in TARGETS}),
        'STRATEGY_WEIGHTS': manager.dict({'bit_flip': 0.3, 'xor_delta': 0.3, 'random': 0.2, 'crossover': 0.1, 'range_shift': 0.1}),
        'MIN_HAMMING': manager.dict({addr: 160 for addr, _ in TARGETS}),
        'HAMMING_HISTORY': manager.dict({addr: manager.list() for addr, _ in TARGETS}),
        'BEST_CANDIDATES': manager.dict()
    })
    load_state(shared_state, lock)
    found_addresses = load_checkpoint()
    remaining_targets = [(addr, h160) for addr, h160 in TARGETS if addr not in found_addresses]

    if not remaining_targets:
        print("All addresses already recovered. Exiting.")
        return

    print(f"Targeting {len(remaining_targets)} remaining addresses.")

    num_workers = min(cpu_count(), len(remaining_targets))
    pool = Pool(num_workers)
    tasks = [(random.randint(1, MAX_K - 1), addr, h160, shared_state, lock)
             for addr, h160 in remaining_targets]

    try:
        last_print = time.time()
        results = pool.map_async(worker_task, tasks)
        while not results.ready():
            time.sleep(1)
            current_time = time.time()
            if current_time - last_print >= 10:
                with lock:
                    for addr in shared_state['BEST_CANDIDATES']:
                        if addr in shared_state['MIN_HAMMING']:
                            k, ham, compressed = shared_state['BEST_CANDIDATES'][addr]
                            p_value = chi_squared_anomaly(shared_state['HAMMING_HISTORY'][addr])
                            print(f"New best for {addr}: {ham} bits off, Key={hex(k)}, Compressed={compressed}, Chi2 p-value={p_value:.4f}")
                    last_print = current_time
                    save_state(shared_state, lock)

        for addr, k, compressed, found, ham in results.get():
            if found:
                print(f"RECOVERED: Address={addr}, Private Key={hex(k)}, Compressed={compressed}")
                found_addresses[addr] = (k, compressed)
                save_checkpoint(found_addresses)
                remaining_targets = [(a, h) for a, h in remaining_targets if a != addr]
                if not remaining_targets:
                    print("All addresses recovered. Exiting.")
                    break
    except KeyboardInterrupt:
        print("Interrupted. Saving checkpoint and state.")
        save_checkpoint(found_addresses)
        save_state(shared_state, lock)
    finally:
        pool.close()
        pool.join()

if __name__ == "__main__":
    main()

Loaded 999 addresses from addresses.txt.
Targeting 999 remaining addresses.
